# Project 9: Comparison of SGD and Adam Optimizers Using a Neural Network on the Iris Dataset

Run this notebook top to bottom in Google Colab or Jupyter (TensorFlow/Keras required). Every experiment uses the same architecture, initial weights, data split, preprocessing, batch size and number of epochs so that the optimizer comparison is fair.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns                      # optional: used only for the pair plot
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42            # random seed for reproducibility
EPOCHS = 100         # same number of epochs for every experiment
BATCH_SIZE = 16      # same batch size for every experiment
CLASS_NAMES = ["Setosa", "Versicolor", "Virginica"]

print("TensorFlow version:", tf.__version__)

## 2. Load the Iris dataset

In [ ]:
iris = load_iris(as_frame=True)          # Iris dataset bundled with scikit-learn
df = iris.frame.copy()                  # features + 'target' column in one DataFrame
df["species"] = df["target"].map(dict(enumerate(CLASS_NAMES)))

print("Shape of dataset:", df.shape)
print(df.head())
print()
print(df["species"].value_counts())
print()
print(df.describe().round(2))

## 3. Dataset visualisation

In [ ]:
# (a) Class distribution
df["species"].value_counts().reindex(CLASS_NAMES).plot(kind="bar", color=["#4c9f70", "#e0a458", "#6a5acd"])
plt.title("Class distribution of the Iris dataset")
plt.ylabel("Number of samples")
plt.xticks(rotation=0)
plt.show()

# (b) Scatter plots of feature pairs
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for sp in CLASS_NAMES:
    d = df[df["species"] == sp]
    axs[0].scatter(d["petal length (cm)"], d["petal width (cm)"], label=sp)
    axs[1].scatter(d["sepal length (cm)"], d["sepal width (cm)"], label=sp)
axs[0].set_xlabel("Petal length (cm)"); axs[0].set_ylabel("Petal width (cm)")
axs[1].set_xlabel("Sepal length (cm)"); axs[1].set_ylabel("Sepal width (cm)")
axs[0].legend()
plt.show()

# (c) Pair plot (optional)
sns.pairplot(df.drop(columns="target"), hue="species")
plt.show()

## 4. Data preprocessing (split + scaling)

In [ ]:
# 1. Separate features (X) and labels (y)
X = iris.data.values.astype("float32")      # shape (150, 4)
y = iris.target.values.astype("int64")      # integer labels 0, 1, 2 (no one-hot encoding needed)

# 2. Hold out 20 % of the data as the TEST set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

# 3. Split the remaining data into TRAINING (80 %) and VALIDATION (20 %) sets
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=SEED, stratify=y_trainval)

# 4. Standardise the features (fit ONLY on the training data)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Training samples  :", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Test samples      :", X_test.shape[0])

## 5. Build the neural network (4 → 16 → 8 → 3)

In [ ]:
def build_model(seed=SEED):
    """Create a fresh 4-16-8-3 network. Re-seeding gives identical initial weights."""
    tf.keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(4,)),
        layers.Dense(16, activation="relu", name="hidden_1"),
        layers.Dense(8, activation="relu", name="hidden_2"),
        layers.Dense(3, activation="softmax", name="output"),
    ])
    return model

build_model().summary()

## 6. Train with SGD

In [ ]:
model_sgd = build_model()

model_sgd.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),   # SGD (Keras default learning rate)
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_sgd = model_sgd.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)
print("SGD training finished.")

## 7. Train with Adam

In [ ]:
model_adam = build_model()        # fresh model: same architecture, same initial weights

model_adam.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001), # Adam (Keras default learning rate)
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_adam = model_adam.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,                # same as SGD
    batch_size=BATCH_SIZE,        # same as SGD
    verbose=0,
)
print("Adam training finished.")

## 8. Evaluate both models on the test set

In [ ]:
results = {}
for name, model, hist in [("SGD", model_sgd, history_sgd), ("Adam", model_adam, history_adam)]:
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    h = hist.history
    results[name] = {
        "Final Training Loss": h["loss"][-1],
        "Final Validation Loss": h["val_loss"][-1],
        "Final Training Accuracy": h["accuracy"][-1],
        "Final Validation Accuracy": h["val_accuracy"][-1],
        "Test Loss": test_loss,
        "Test Accuracy": test_acc,
    }
    print(f"{name:5s} -> Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

print()
print(pd.DataFrame(results).round(4))

### Training history at selected epochs

In [ ]:
def history_table(h, epochs=(1, 10, 25, 50, 75, 100)):
    t = pd.DataFrame({
        "Epoch": epochs,
        "Train Loss": [h["loss"][e - 1] for e in epochs],
        "Train Acc":  [h["accuracy"][e - 1] for e in epochs],
        "Val Loss":   [h["val_loss"][e - 1] for e in epochs],
        "Val Acc":    [h["val_accuracy"][e - 1] for e in epochs],
    })
    return t.round(4).to_string(index=False)

print("SGD (learning rate = 0.01)");  print(history_table(history_sgd.history));  print()
print("Adam (learning rate = 0.001)"); print(history_table(history_adam.history))

## 9. Training behaviour: loss and accuracy curves

In [ ]:
def compare_plot(key, title, ylabel):
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, EPOCHS + 1), history_sgd.history[key], label="SGD (lr=0.01)")
    plt.plot(range(1, EPOCHS + 1), history_adam.history[key], label="Adam (lr=0.001)")
    plt.xlabel("Epochs"); plt.ylabel(ylabel); plt.title(title)
    plt.legend(); plt.grid(True); plt.show()

compare_plot("loss",         "Training loss: SGD vs Adam",       "Loss")
compare_plot("val_loss",     "Validation loss: SGD vs Adam",     "Loss")
compare_plot("accuracy",     "Training accuracy: SGD vs Adam",   "Accuracy")
compare_plot("val_accuracy", "Validation accuracy: SGD vs Adam", "Accuracy")

## 10. Predictions

In [ ]:
pred_sgd  = np.argmax(model_sgd.predict(X_test, verbose=0),  axis=1)
pred_adam = np.argmax(model_adam.predict(X_test, verbose=0), axis=1)

comparison = pd.DataFrame({
    "True":  [CLASS_NAMES[i] for i in y_test],
    "SGD":   [CLASS_NAMES[i] for i in pred_sgd],
    "Adam":  [CLASS_NAMES[i] for i in pred_adam],
})
print(comparison)

## 11. Confusion matrices

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in zip(axs, ["SGD", "Adam"], [pred_sgd, pred_adam]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion matrix: {name}")
plt.tight_layout(); plt.show()

## 12. Classification reports

In [ ]:
print("===== SGD =====")
print(classification_report(y_test, pred_sgd, target_names=CLASS_NAMES, digits=4))
print("===== Adam =====")
print(classification_report(y_test, pred_adam, target_names=CLASS_NAMES, digits=4))

## 13. Hyperparameter experiment: SGD learning rates

In [ ]:
lr_rows = []
plt.figure(figsize=(7, 4))
for lr in [0.001, 0.01, 0.1]:
    m = build_model()                                    # same initial weights every time
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    t_loss, t_acc = m.evaluate(X_test, y_test, verbose=0)
    lr_rows.append({"SGD Learning Rate": lr,
                    "Final Train Loss": h.history["loss"][-1],
                    "Final Val Loss": h.history["val_loss"][-1],
                    "Test Loss": t_loss, "Test Accuracy": t_acc})
    plt.plot(h.history["loss"], label=f"lr = {lr}")

plt.xlabel("Epochs"); plt.ylabel("Training loss")
plt.title("SGD with different learning rates"); plt.legend(); plt.grid(True); plt.show()
print(pd.DataFrame(lr_rows).round(4))

## 14. Supplementary: repeated runs with different seeds

In [ ]:
def run_once(opt_name, seed):
    m = build_model(seed)                 # different initial weights and shuffling for each seed
    opt = (keras.optimizers.SGD(learning_rate=0.01) if opt_name == "SGD"
           else keras.optimizers.Adam(learning_rate=0.001))
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(X_train, y_train, validation_data=(X_val, y_val),
          epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    return m.evaluate(X_test, y_test, verbose=0)          # (test_loss, test_accuracy)

for opt_name in ["SGD", "Adam"]:
    outs = np.array([run_once(opt_name, s) for s in range(10)])
    print(f"{opt_name:5s} test accuracy: mean = {outs[:,1].mean():.4f}, std = {outs[:,1].std():.4f}, "
          f"min = {outs[:,1].min():.4f}, max = {outs[:,1].max():.4f}")